# 03 — CNN Baseline (Floor for YOLO benchmark, SCRUM-7)

**Model Lead task.** This notebook trains a small **from-scratch CNN classifier** (no pretrained weights) on the merged dental X-ray dataset Varsha prepared in `01_eda.ipynb`. Its job is to be the **floor** — the final YOLOv8 detector must beat these numbers, or the extra complexity of detection isn't earning its keep.

**Why not transfer learning here:** a pretrained ImageNet backbone would set an artificially strong floor. Keeping this model simple and trained from scratch — same spirit as the CNNs you built in `Lab1C` — gives an honest baseline.

**Design decision (assumption — flag if wrong):** the merged dataset is labeled for *detection* (bounding boxes), but a classifier needs one label per image. Each image is assigned a single dominant class by priority: `Cavity > Crown > Impacted Tooth > Filling > No Finding`. Rationale: rarer/more clinically significant findings take priority over the very common `Filling` class when an image has more than one type of box.

| Section | What happens |
|---|---|
| 1 | Build single-label index from merged YOLO labels |
| 2 | Load images into a `tf.data` pipeline |
| 3 | Define the CNN (3 conv blocks, trained from scratch) |
| 4 | Train, with class weights for the imbalance Varsha flagged |
| 5 | Evaluate: accuracy, confusion matrix, most-confused pair, misclassified samples |
| 6 | Log metrics + save the checkpoint |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(42)
tf.random.set_seed(42)

import sys
sys.path.append('..')  # so `import src...` works whether this runs from notebooks/ or repo root

DATA_DIR = Path('../data/processed')  # output of 01_eda.ipynb
IMG_SIZE = 128
BATCH_SIZE = 32

# 4 pathology classes + a 5th 'No Finding' class for images with no boxes at all
CLASS_NAMES = ['Cavity', 'Filling', 'Crown', 'Impacted Tooth', 'No Finding']
PRIORITY = ['Cavity', 'Crown', 'Impacted Tooth', 'Filling']  # highest priority first; No Finding is the fallback
NUM_CLASSES = len(CLASS_NAMES)

print('Classes:', CLASS_NAMES)

Classes: ['Cavity', 'Filling', 'Crown', 'Impacted Tooth', 'No Finding']


## 1. Build a single-label index from the merged YOLO labels

Reads each split's `labels/*.txt`, applies the priority rule above, and produces `(image_path, class_id)` pairs.

In [ ]:
!git clone https://github.com/neuroarcane/dental-cavity-detector.git


Cloning into 'dental-cavity-detector'...
remote: Enumerating objects: 27719, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 27719 (delta 46), reused 78 (delta 29), pack-reused 27607 (from 3)
Receiving objects: 100% (27719/27719), 899.11 MiB | 33.82 MiB/s, done.
Resolving deltas: 100% (55/55), done.
Updating files: 100% (30039/30039), done.


In [ ]:
%cd dental-cavity-detector
!pip install -r requirements.txt

from pathlib import Path
data_root = Path('/content/dental_data')
data_root.mkdir(parents=True, exist_ok=True)
!cp -r "data/raw/Dental X-ray.v1i.yolov11" /content/dental_data/
!cp -r "data/raw/Dental X-Ray Panoramic Dataset" /content/dental_data/
(data_root / "Dental X-ray.v1i.yolov11.zip.extracted").touch()
(data_root / "Dental X-Ray Panoramic Dataset.zip.extracted").touch()

/content/dental-cavity-detector
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 19.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 36.8

In [ ]:
!ls /content/dental_data
!ls /content/dental-cavity-detector/data/raw

'Dental X-Ray Panoramic Dataset'
'Dental X-Ray Panoramic Dataset.zip.extracted'
'Dental X-ray.v1i.yolov11'
'Dental X-ray.v1i.yolov11.zip.extracted'
'Dental X-Ray Panoramic Dataset'  'Dental X-ray.v1i.yolov11'


In [ ]:
# Class-name-to-id mapping must match data/processed/data.yaml exactly — confirm before running.
import yaml
with open(DATA_DIR / 'data.yaml') as f:
    data_yaml = yaml.safe_load(f)
print(data_yaml)

yaml_class_names = data_yaml['names']  # id -> name, in YOLO's order
name_to_id = {name: i for i, name in enumerate(yaml_class_names)}
print(name_to_id)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/data.yaml'

In [ ]:
def build_singlelabel_index(split_dir: Path):
    images_dir = split_dir / 'images'
    labels_dir = split_dir / 'labels'
    rows = []
    for img_path in sorted(images_dir.glob('*.*')):
        label_path = labels_dir / f'{img_path.stem}.txt'
        present = set()
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                if line.strip():
                    cls_id = int(line.split()[0])
                    present.add(yaml_class_names[cls_id])

        chosen = 'No Finding'
        for cls in PRIORITY:
            if cls in present:
                chosen = cls
                break

        rows.append({'path': str(img_path), 'class_name': chosen, 'class_id': CLASS_NAMES.index(chosen)})
    return pd.DataFrame(rows)

train_df = build_singlelabel_index(DATA_DIR / 'train')
val_df   = build_singlelabel_index(DATA_DIR / 'valid')
test_df  = build_singlelabel_index(DATA_DIR / 'test')

print('Train class counts:')
print(train_df['class_name'].value_counts())
print('\nVal class counts:')
print(val_df['class_name'].value_counts())

**Check before continuing:** per Varsha's PR notes, the smaller source dataset contributes zero `Crown` labels. If `val_df` or `test_df` above shows 0 for any class, the split isn't stratified by source and metrics for that class will be meaningless — flag it back to her rather than training on it blind.

## 2. `tf.data` pipeline

In [ ]:
def make_dataset(df: pd.DataFrame, shuffle: bool = False):
    paths = df['path'].values
    labels = df['class_id'].values

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = img / 255.0
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=42)
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

# Sanity check: look at one batch
for imgs, labels in train_ds.take(1):
    print('Batch shape:', imgs.shape, 'Label shape:', labels.shape)
    fig, axes = plt.subplots(1, 5, figsize=(12, 3))
    for i, ax in enumerate(axes):
        ax.imshow(imgs[i])
        ax.set_title(CLASS_NAMES[labels[i].numpy()])
        ax.axis('off')
    plt.show()

## 3. Model — from-scratch CNN, 3 conv blocks

Same pattern as `Lab1C` Exercise 3 (conv → pool, repeated, then flatten → dense → output) but scaled up for larger, RGB, real-world images instead of 28×28 MNIST digits.

In [ ]:
from tensorflow.keras import layers, models

def build_cnn_baseline(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),

        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model

model = build_cnn_baseline()
model.summary()

## 4. Train

Class weights offset the imbalance Varsha documented (`Filling` ~5x `Cavity`/`Crown`) so the model can't just learn to always predict the majority class.

In [ ]:
class_ids = np.arange(NUM_CLASSES)
weights = compute_class_weight(class_weight='balanced', classes=class_ids, y=train_df['class_id'].values)
class_weight = dict(zip(class_ids, weights))
print('Class weights:', {CLASS_NAMES[i]: round(w, 2) for i, w in class_weight.items()})

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
)

In [ ]:
# Training curves — same idea as any lab: plot loss/accuracy vs epoch
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()
plt.savefig('../models/runs/cnn_baseline_training_curves.png')
plt.show()

## 5. Evaluate — same shape as `Lab1C` Exercise 5

Accuracy on its own hides *which* mistakes the model makes — the confusion matrix, most-confused pair, and misclassified samples below make that visible, exactly as in the lab exercise.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss: {test_loss:.4f}')

y_true = test_df['class_id'].values
y_pred_probs = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('CNN Baseline — Confusion Matrix')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.colorbar(im)
plt.tight_layout()
plt.savefig('../models/runs/cnn_baseline_confusion_matrix.png')
plt.show()

# Most confused pair — same trick as the lab exercise
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
worst = np.unravel_index(np.argmax(cm_off), cm_off.shape)
print(f'Most confused: true {CLASS_NAMES[worst[0]]} predicted as {CLASS_NAMES[worst[1]]} ({cm_off[worst]} times)')

In [ ]:
# Plot a few misclassified test images
wrong_idx = np.where(y_pred != y_true)[0][:5]

fig, axes = plt.subplots(1, len(wrong_idx), figsize=(3 * len(wrong_idx), 3))
for ax, i in zip(np.atleast_1d(axes), wrong_idx):
    img = tf.io.read_file(test_df.iloc[i]['path'])
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]) / 255.0
    ax.imshow(img)
    ax.set_title(f"pred: {CLASS_NAMES[y_pred[i]]}\ntrue: {CLASS_NAMES[y_true[i]]}", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Log metrics + save checkpoint

This is the number Iva (eval lead) and Temirlan (tuning lead) will compare the final YOLOv8 model against.

In [ ]:
import json
from pathlib import Path

Path('../models/runs').mkdir(parents=True, exist_ok=True)

pd.DataFrame(history.history).to_csv('../models/runs/cnn_baseline_history.csv', index=False)

summary = {
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'most_confused_true': CLASS_NAMES[worst[0]],
    'most_confused_pred': CLASS_NAMES[worst[1]],
    'most_confused_count': int(cm_off[worst]),
    'class_names': CLASS_NAMES,
    'epochs': len(history.history['loss']),
}
with open('../models/runs/cnn_baseline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

model.save('../models/runs/cnn_baseline.keras')
print('Saved: cnn_baseline_history.csv, cnn_baseline_summary.json, cnn_baseline.keras, confusion matrix + training curve PNGs')
print('\nFLOOR METRIC for YOLO to beat: test_accuracy =', round(test_acc, 4))

## Notes for the team

- **Floor metric:** test accuracy above (see printed summary). The final YOLOv8 model should be evaluated on a comparable basis — e.g. per-image "did the detector find at least the dominant pathology" — for an apples-to-apples comparison, since YOLO's native metric (mAP) isn't directly the same unit.
- **Known limitation:** the single-label priority scheme discards information for images with multiple pathologies. This is intentional for a *baseline* — not a design choice to carry into the final detector.
- **Class imbalance and source-stratification issues** (per Varsha's PR) directly affect this model's per-class recall, especially for `Crown`. If `Crown` recall is near zero, check whether the test split actually contains `Crown` examples before concluding the model failed.

In [7]:
!git clone https://github.com/neuroarcane/dental-cavity-detector.git
%cd dental-cavity-detector
!git checkout main
!git pull origin main

Cloning into 'dental-cavity-detector'...
remote: Enumerating objects: 27794, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 27794 (delta 21), reused 27 (delta 11), pack-reused 27740 (from 2)
Receiving objects: 100% (27794/27794), 994.75 MiB | 29.34 MiB/s, done.
Resolving deltas: 100% (84/84), done.
Updating files: 100% (30081/30081), done.
/content/dental-cavity-detector
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/neuroarcane/dental-cavity-detector
 * branch              main       -> FETCH_HEAD
Already up to date.


In [8]:
!pwd
!ls notebooks/

/content/dental-cavity-detector
01_eda.ipynb		05_cnn_overfitting_experiment.ipynb   README.md
02_preprocessing.ipynb	06_batch_size_comparison.ipynb
03_evaluation.ipynb	07_model_testing_visualization.ipynb
